# reg_one_variate

> 对应代码：`EconometricsCode/code_in_notes/Chap.1/reg_one_variate.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.1`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [1]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.1")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Single-user 16-core  perpetual
Serial number: 501806307019
  Licensed to: Keep calm and DO your research
               Jichun Si

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


读入中国家庭金融调查（CHFS）2017 年家庭数据，并做两步清理：剔除负收入的异常样本，以及消费或收入存在截断（censor）标记的样本，保证回归样本的质量。

In [2]:
%%stata
use ../datasets/chfs2017_hh.dta, clear
drop if total_income < 0
drop if max(censor_total_consump, censor_total_income)


. use ../datasets/chfs2017_hh.dta, clear

. drop if total_income < 0
(566 observations deleted)

. drop if max(censor_total_consump, censor_total_income)
(41 observations deleted)

. 


做本章最基本的一元线性回归：以总收入解释总消费。得到的斜率即边际消费倾向的估计——收入每增加一元，消费平均增加多少；outreg2 把结果输出为 LaTeX 表格。

In [3]:
%%stata
reg total_consump total_income
outreg2 using reg_one_variate.tex, replace


. reg total_consump total_income

      Source |       SS           df       MS      Number of obs   =    39,404
-------------+----------------------------------   F(1, 39402)     =   8337.56
       Model |  2.6870e+13         1  2.6870e+13   Prob > F        =    0.0000
    Residual |  1.2698e+14    39,402  3.2227e+09   R-squared       =    0.1746
-------------+----------------------------------   Adj R-squared   =    0.1746
       Total |  1.5385e+14    39,403  3.9046e+09   Root MSE        =     56769

------------------------------------------------------------------------------
total_cons~p | Coefficient  Std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
total_income |   .1617166   .0017711    91.31   0.000     .1582452    .1651879
       _cons |   44221.38   326.1484   135.59   0.000     43582.12    44860.64
------------------------------------------------------------------------------

. outreg2 using

predict 生成消费的拟合值（预测值）ŷᵢ=x̂β，即回归直线上的点，并加上中文标签。

In [4]:
%%stata
predict pred_consump
label variable pred_consump "预测的消费"


. predict pred_consump
(option xb assumed; fitted values)

. label variable pred_consump "预测的消费"

. 


先按收入排序（画折线图所必需），再把散点图与拟合直线叠加：scatter 展示原始数据点，line 画出回归直线，直观呈现一元回归"用一条直线拟合散点"的几何含义。

In [5]:
%%stata
sort total_income
twoway (scatter total_consump total_income) (line pred_consump total_income)


. sort total_income

. twoway (scatter total_consump total_income) (line pred_consump total_income)

. 
